# 09 - Data Segmentation Analysis

This notebook groups customers and products into meaningful business segments.

Focus areas:
- Product cost segmentation
- Customer value segmentation
- Customer lifecycle segmentation
- Revenue contribution by segment
- Product sales performance segments

In [0]:
%sql
/*
Product Cost Segmentation

Purpose:
    Group products into cost ranges to understand the price structure
    of the product catalog.
*/

WITH product_segments AS (
    SELECT
        product_key,
        product_name,
        COALESCE(category, 'Unknown') AS category,
        cost,
        CASE 
            WHEN cost < 100 THEN 'Below 100'
            WHEN cost BETWEEN 100 AND 500 THEN '100-500'
            WHEN cost BETWEEN 501 AND 1000 THEN '501-1000'
            ELSE 'Above 1000'
        END AS cost_range
    FROM datawarehouseanalytics_gold.dim_products
)

SELECT 
    cost_range,
    COUNT(product_key) AS total_products,
    ROUND(AVG(cost), 2) AS avg_cost,
    MIN(cost) AS min_cost,
    MAX(cost) AS max_cost
FROM product_segments
GROUP BY cost_range
ORDER BY total_products DESC;

In [0]:
%sql
/*
Product Cost Segmentation by Category

Purpose:
    Compare cost ranges across product categories to understand how
    pricing differs by product group.
*/

WITH product_segments AS (
    SELECT
        product_key,
        COALESCE(category, 'Unknown') AS category,
        cost,
        CASE 
            WHEN cost < 100 THEN 'Below 100'
            WHEN cost BETWEEN 100 AND 500 THEN '100-500'
            WHEN cost BETWEEN 501 AND 1000 THEN '501-1000'
            ELSE 'Above 1000'
        END AS cost_range
    FROM datawarehouseanalytics_gold.dim_products
)

SELECT
    category,
    cost_range,
    COUNT(product_key) AS total_products,
    ROUND(
        COUNT(product_key) * 100.0 
        / SUM(COUNT(product_key)) OVER (PARTITION BY category),
        2
    ) AS percentage_within_category
FROM product_segments
GROUP BY category, cost_range
ORDER BY category, total_products DESC;

In [0]:
%sql
/*
Customer Spending Segmentation

Purpose:
    Segment customers based on spending value and purchase lifespan.

Segment Logic:
    VIP: Customers with at least 12 months of purchase history and spending above 5000.
    Regular: Customers with at least 12 months of purchase history and spending up to 5000.
    New: Customers with less than 12 months of purchase history.
*/

WITH customer_spending AS (
    SELECT
        c.customer_key,
        c.customer_number,
        c.first_name,
        c.last_name,
        c.country,
        SUM(f.sales_amount) AS total_spending,
        COUNT(DISTINCT f.order_number) AS total_orders,
        MIN(f.order_date) AS first_order_date,
        MAX(f.order_date) AS last_order_date,
        ROUND(MONTHS_BETWEEN(MAX(f.order_date), MIN(f.order_date)), 0) AS customer_lifespan_months
    FROM datawarehouseanalytics_gold.fact_sales f
    LEFT JOIN datawarehouseanalytics_gold.dim_customers c
        ON f.customer_key = c.customer_key
    GROUP BY
        c.customer_key,
        c.customer_number,
        c.first_name,
        c.last_name,
        c.country
),

segmented_customers AS (
    SELECT
        *,
        CASE 
            WHEN customer_lifespan_months >= 12 AND total_spending > 5000 THEN 'VIP'
            WHEN customer_lifespan_months >= 12 AND total_spending <= 5000 THEN 'Regular'
            ELSE 'New'
        END AS customer_segment
    FROM customer_spending
)

SELECT 
    customer_segment,
    COUNT(customer_key) AS total_customers,
    SUM(total_spending) AS total_revenue,
    ROUND(AVG(total_spending), 2) AS avg_customer_spending,
    ROUND(AVG(total_orders), 2) AS avg_orders_per_customer
FROM segmented_customers
GROUP BY customer_segment
ORDER BY total_revenue DESC;

In [0]:
%sql
/*
Customer Segment by Country

Purpose:
    Analyze how customer value segments are distributed across countries.
*/

WITH customer_spending AS (
    SELECT
        c.customer_key,
        COALESCE(c.country, 'Unknown') AS country,
        SUM(f.sales_amount) AS total_spending,
        MIN(f.order_date) AS first_order_date,
        MAX(f.order_date) AS last_order_date,
        ROUND(MONTHS_BETWEEN(MAX(f.order_date), MIN(f.order_date)), 0) AS customer_lifespan_months
    FROM datawarehouseanalytics_gold.fact_sales f
    LEFT JOIN datawarehouseanalytics_gold.dim_customers c
        ON f.customer_key = c.customer_key
    GROUP BY c.customer_key, COALESCE(c.country, 'Unknown')
),

segmented_customers AS (
    SELECT
        *,
        CASE 
            WHEN customer_lifespan_months >= 12 AND total_spending > 5000 THEN 'VIP'
            WHEN customer_lifespan_months >= 12 AND total_spending <= 5000 THEN 'Regular'
            ELSE 'New'
        END AS customer_segment
    FROM customer_spending
)

SELECT
    country,
    customer_segment,
    COUNT(customer_key) AS total_customers,
    SUM(total_spending) AS total_revenue,
    ROUND(
        COUNT(customer_key) * 100.0 
        / SUM(COUNT(customer_key)) OVER (PARTITION BY country),
        2
    ) AS segment_percentage_within_country
FROM segmented_customers
GROUP BY country, customer_segment
ORDER BY country, total_revenue DESC;

In [0]:
%sql
/*
Customer RFM-Style Segmentation

Purpose:
    Segment customers using recency, frequency, and monetary value.
    Recency is calculated against the latest order date in the dataset.
*/

WITH latest_order AS (
    SELECT MAX(order_date) AS max_order_date
    FROM datawarehouseanalytics_gold.fact_sales
),

customer_metrics AS (
    SELECT
        c.customer_key,
        c.customer_number,
        c.first_name,
        c.last_name,
        COALESCE(c.country, 'Unknown') AS country,
        DATEDIFF(l.max_order_date, MAX(f.order_date)) AS recency_days,
        COUNT(DISTINCT f.order_number) AS frequency_orders,
        SUM(f.sales_amount) AS monetary_value
    FROM datawarehouseanalytics_gold.fact_sales f
    LEFT JOIN datawarehouseanalytics_gold.dim_customers c
        ON f.customer_key = c.customer_key
    CROSS JOIN latest_order l
    GROUP BY
        c.customer_key,
        c.customer_number,
        c.first_name,
        c.last_name,
        COALESCE(c.country, 'Unknown'),
        l.max_order_date
),

rfm_segments AS (
    SELECT
        *,
        CASE
            WHEN recency_days <= 180 AND frequency_orders >= 5 AND monetary_value >= 5000 THEN 'Champions'
            WHEN recency_days <= 365 AND monetary_value >= 3000 THEN 'Loyal Customers'
            WHEN recency_days > 365 AND monetary_value >= 3000 THEN 'At Risk High Value'
            WHEN frequency_orders = 1 THEN 'One-Time Buyers'
            ELSE 'Standard Customers'
        END AS rfm_segment
    FROM customer_metrics
)

SELECT
    rfm_segment,
    COUNT(customer_key) AS total_customers,
    ROUND(AVG(recency_days), 2) AS avg_recency_days,
    ROUND(AVG(frequency_orders), 2) AS avg_frequency_orders,
    ROUND(AVG(monetary_value), 2) AS avg_monetary_value,
    SUM(monetary_value) AS total_revenue
FROM rfm_segments
GROUP BY rfm_segment
ORDER BY total_revenue DESC;